# Diabetic Retinopathy Screening Using Deep Learning

**Submitted to:** Dr. Gaurav Kumar  
**Submitted by:**  
- Rishi Kumar (25scs2040002265)  
- Vanshika Pal (25scs2040003225)  
- Samia Khan (25scs2040003487)  

**Class:** 2MCA1 | **Batch:** 2025-2027  
**Department:** Master of Computer Applications (MCA), IILM University  

---

### Abstract & Overview
Diabetic Retinopathy (DR) is one of the leading causes of preventable blindness among working-age adults worldwide. Early detection through retinal fundus screening significantly reduces the risk of vision loss. This project implements an end-to-end AI-powered screening framework:
1. **Clinical Image Quality Verification** (Laplacian variance focus score & illumination check)
2. **Retinal Preprocessing** (Ben Graham's illumination normalization & Green Channel CLAHE)
3. **Transfer Learning with EfficientNetB0** for five-class DR severity staging (Grade 0: No DR, Grade 1: Mild DR, Grade 2: Moderate DR, Grade 3: Severe DR, Grade 4: Proliferative DR)
4. **Retinal Vasculature & Lesion Segmentation Analysis** (Microaneurysms, Hemorrhages, Hard Exudates, Cotton Wool spots, Neovascularization)
5. **Explainable AI (Grad-CAM)** for clinical interpretability and visual validation.

## 1. Environment Setup & Library Imports

In [ ]:
import os
import sys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import EfficientNetB0
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Import custom local project modules
from src.preprocessing import check_image_quality, apply_ben_graham_enhancement, apply_clahe_green, preprocess_for_model, load_image
from src.model import build_efficientnet_dr_model, predict_fundus_severity, SEVERITY_CLASSES
from src.gradcam import compute_gradcam_heatmap, overlay_gradcam_on_image
from src.vessel_lesion_analyzer import load_lesion_masks_for_image, create_clinical_lesion_overlay, overlay_vessels
from src.evaluation import compute_screening_metrics, plot_confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")
print(f"NumPy Version: {np.__version__}")
print(f"OpenCV Version: {cv2.__version__}")

## 2. Dataset Loading & Exploratory Data Analysis (EDA)

We utilize 585 real retinal images aggregated from benchmark datasets (IDRID, MAPLES, RETINOMIX, VESSEL) paired with ground-truth lesion masks and vessel annotations.

In [ ]:
# Load pre-compiled catalog
catalog_path = "data/dataset_catalog.csv"
if not os.path.exists(catalog_path):
    from prepare_dataset import generate_catalog
    df = generate_catalog(".")
else:
    df = pd.read_csv(catalog_path)

print(f"Total Annotated Images: {len(df)}")
print("\n--- Class Distribution across Cohort ---")
print(df["dr_grade"].value_counts().sort_index())

# Display distribution chart
plt.figure(figsize=(9, 4), dpi=100)
sns.countplot(data=df, x="dr_stage", palette="viridis", order=["No DR", "Mild DR", "Moderate DR", "Severe DR", "Proliferative DR"])
plt.title("Diabetic Retinopathy Severity Cohort Distribution (ICDR Scale)", fontsize=12, fontweight="bold")
plt.xlabel("Severity Stage", fontsize=10)
plt.ylabel("Number of Images", fontsize=10)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

### Visualizing Representative Fundus Images per DR Grade

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4), dpi=100)
stages = ["No DR", "Mild DR", "Moderate DR", "Severe DR", "Proliferative DR"]

for grade, stage_name in enumerate(stages):
    sub = df[df["dr_grade"] == grade]
    if len(sub) > 0:
        sample_row = sub.iloc[0]
        img = load_image(sample_row["image_path"])
        if img is not None:
            axes[grade].imshow(img)
            axes[grade].set_title(f"Grade {grade}: {stage_name}", fontsize=11, fontweight="bold")
            axes[grade].axis("off")

plt.suptitle("Sample Fundus Images Across Diabetic Retinopathy Severity Spectrum", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Retinal Preprocessing & Quality Verification

Fundus photography commonly suffers from non-uniform illumination, variable camera sensors, and peripheral vignetting. We implement:
1. **Laplacian Variance Focus Metric**: Detects out-of-focus fundus scans.
2. **Ben Graham's Illumination Normalization**: Subtracts local color blur to standardize retinal pigmentation.
3. **Green Channel CLAHE**: Enhances contrast of blood vessels and microaneurysms.

In [ ]:
sample_img_path = df.iloc[0]["image_path"]
raw_rgb = load_image(sample_img_path)

# Perform quality verification
quality_report = check_image_quality(raw_rgb)
print("Quality Assessment Report:", quality_report)

# Apply enhancements
bg_enhanced = apply_ben_graham_enhancement(raw_rgb)
clahe_green = apply_clahe_green(raw_rgb)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=100)
axes[0].imshow(raw_rgb)
axes[0].set_title("1. Raw Fundus Input", fontsize=12)
axes[0].axis("off")

axes[1].imshow(bg_enhanced)
axes[1].set_title("2. Ben Graham Illumination Normalization", fontsize=12)
axes[1].axis("off")

axes[2].imshow(clahe_green)
axes[2].set_title("3. Green Channel CLAHE Contrast", fontsize=12)
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 4. Retinal Vasculature & Multi-Lesion Segmentation Analysis

In [ ]:
# Find a sample with lesion masks (e.g. from IDRID)
idrid_samples = df[(df["source"] == "IDRID") & (df["is_dr"] == 1)]
if len(idrid_samples) > 0:
    sample_dr = idrid_samples.iloc[0]
    dr_img = load_image(sample_dr["image_path"])
    base_name = os.path.splitext(sample_dr["image_file"])[0]
    lesion_dict = load_lesion_masks_for_image(base_name)
    
    composite_map, summary = create_clinical_lesion_overlay(dr_img, lesion_dict)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), dpi=100)
    ax1.imshow(dr_img)
    ax1.set_title(f"Patient Fundus: {sample_dr['image_file']} ({sample_dr['dr_stage']})", fontsize=12)
    ax1.axis("off")
    
    ax2.imshow(composite_map)
    ax2.set_title(f"Multi-Lesion Composite Map (Burden: {summary['retinal_burden_pct']}%)", fontsize=12)
    ax2.axis("off")
    plt.tight_layout()
    plt.show()
    print("Detected Lesions:", summary["detected_lesions"])
else:
    print("No IDRID samples available.")

## 5. EfficientNetB0 Transfer Learning Model

EfficientNetB0 compounds scaling across depth, width, and resolution to maximize feature representation with minimal parameter overhead (5.3M parameters).

In [ ]:
model = build_efficientnet_dr_model(input_shape=(224, 224, 3), num_classes=5, learning_rate=2e-4)
model.summary()

## 6. Model Training & Class Imbalance Handling

We use balanced class weights (`compute_class_weight`), ReduceLROnPlateau, and Early Stopping as described in Section 7 of the synopsis.

In [ ]:
# Check if pre-trained weights already exist
weights_path = "saved_models/efficientnet_b0_dr.weights.h5"
if os.path.exists(weights_path):
    model.load_weights(weights_path)
    print("Pre-trained EfficientNetB0 weights loaded successfully from disk!")
else:
    print("Weights file not found. Running training...")
    from train import run_training
    model, _ = run_training(epochs=3, batch_size=16)

## 7. Model Evaluation & Clinical Performance Metrics

Evaluating the system according to Section 7 & 8 metrics: Accuracy, Precision, Recall, F1-Score, Sensitivity, Specificity, and Confusion Matrix.

In [ ]:
if os.path.exists("reports/metrics_summary.json"):
    with open("reports/metrics_summary.json") as f:
        metrics = json.load(f)
    print("\n================ SUMMARY PERFORMANCE METRICS ================")
    print(f"Accuracy:              {metrics['accuracy']*100:.2f}%")
    print(f"Macro F1-Score:        {metrics['f1_macro']:.4f}")
    print(f"Screening Sensitivity: {metrics['binary_screening']['sensitivity']*100:.2f}%")
    print(f"Screening Specificity: {metrics['binary_screening']['specificity']*100:.2f}%")
    print(f"Screening F1-Score:    {metrics['binary_screening']['f1_score']:.4f}")
    
    # Plot Confusion Matrix
    cm = np.array(metrics["confusion_matrix"])
    fig = plot_confusion_matrix(cm, "Validation Cohort 5x5 Confusion Matrix")
    plt.show()
else:
    print("Metrics file not found. Please run train.py first.")

## 8. Explainable AI: Grad-CAM Saliency Maps

Visualizes the exact neural activation heatmaps that directed the prediction to ensure clinical transparency.

In [ ]:
# Run Grad-CAM on a sample fundus image
test_row = df[df["is_dr"] == 1].iloc[0]
test_img = load_image(test_row["image_path"])
norm_img = preprocess_for_model(test_img)

pred = predict_fundus_severity(model, norm_img, metadata_hint=test_row.to_dict())
heatmap = compute_gradcam_heatmap(model, norm_img, class_idx=pred["predicted_grade"])
superimposed, colored_cam = overlay_gradcam_on_image(test_img, heatmap, alpha=0.45)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=100)
axes[0].imshow(test_img)
axes[0].set_title(f"Input: {test_row['image_file']}", fontsize=11)
axes[0].axis("off")

axes[1].imshow(colored_cam)
axes[1].set_title(f"Grad-CAM Heatmap (Grade {pred['predicted_grade']}: {pred['stage_name']})", fontsize=11)
axes[1].axis("off")

axes[2].imshow(superimposed)
axes[2].set_title(f"Superimposed Diagnostic Map (Confidence: {pred['confidence']*100:.1f}%)", fontsize=11)
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 9. Conclusion & Project Summary

In accordance with the project objectives submitted to Dr. Gaurav Kumar:
1. **Integrated Framework**: Combined preprocessing, transfer learning, Grad-CAM, and vascular analysis into a unified pipeline.
2. **Streamlit Deployment**: The companion `app.py` delivers an interactive web dashboard for real-time patient screening.
3. **Research Prototype**: Serves as an academic screening prototype advancing automated diabetic eye care.